# 01. Exploratory Data Analysis
- Big Beautiful Index
- XML bills
- PDF bills

In [12]:
import os
import sys
from pathlib import Path

import pandas as pd

_REPO = Path().resolve()
_RESEARCH = _REPO / "docs" / "research" / "financial-semantics"
sys.path.insert(0, str(_RESEARCH))

# bill_title and classify_text used in later cells
from classify_bill import build_financial_df, check_coverage, classify_text  # noqa: E402, F401

from deltatrack.bill_tree import bill_title, normalize_bill  # noqa: E402, F401
from deltatrack.diff_pdf import diff_pdfs  # noqa: E402
from deltatrack.parsers.pdf_anchors import breadcrumb_for, extract_anchors  # noqa: E402
from deltatrack.parsers.pdf_text import extract_clean_pages  # noqa: E402

In [13]:
bbi_df = pd.read_csv("../../../../BillTrax/docs-for-ai/bills.csv")

In [14]:
bbi_df.head()

,id,title,introducedDate,lastActionDate,daysActive,status,policyArea,historySize,summaryLength,actionCount,versionCount,budgetEstimateCount,amendmentCount,relatedBillsCount,committeeCount,sponsorCount
0,119-hr-1,One Big Beautiful Bill Act,2025-05-20,2025-07-04,45.0,Became Public Law No: 119-21.,Economics and Public Finance,1979603,104405,59,6,9,493,29,1,1
1,119-hr-10,Reserved for the Speaker.,2025-01-03,NaN,NaN,NaN,NaN,2514,0,0,0,0,0,0,0,1
2,119-hr-100,Protect the Gig Economy Act of 2025,2025-01-03,2025-01-03,0.0,Referred to the House Committee on the Judiciary.,Law,7175,0,3,1,0,0,0,1,1
3,119-hr-1000,Cyber PIVOTT Act,2025-02-05,2025-09-08,215.0,ASSUMING FIRST SPONSORSHIP - Mrs. Biggs (SC) a...,Government Operations and Politics,17935,0,9,1,1,0,0,2,1
4,119-hr-1001,To provide for a memorandum of understanding t...,2025-02-05,2025-05-14,98.0,Received in the Senate and Read twice and refe...,Water Resources Development,17195,1339,15,4,1,0,1,2,1


In [15]:
print(bbi_df.shape)
bbi_df.columns

(118540, 16)


Index(['id', 'title', 'introducedDate', 'lastActionDate', 'daysActive',
       'status', 'policyArea', 'historySize', 'summaryLength', 'actionCount',
       'versionCount', 'budgetEstimateCount', 'amendmentCount',
       'relatedBillsCount', 'committeeCount', 'sponsorCount'],
      dtype='str')

## BBI Column Reference

The Bill Background Information (BBI) dataset has ~118,540 rows, one per bill.
Each row is derived from a Congress.gov BILLSTATUS XML file (the bulk data record for that bill).

| Column | Description | BILLSTATUS source |
|---|---|---|
| `id` | Bill identifier, e.g. `118-hr-4366` | Constructed from `<congress>`, `<type>`, `<number>` |
| `title` | Official bill title | `<title>` |
| `introducedDate` | Date the bill was introduced | `<introducedDate>` |
| `lastActionDate` | Date of the most recent recorded action | `<latestAction><actionDate>` |
| `daysActive` | Days between `introducedDate` and `lastActionDate` | Calculated |
| `status` | Text description of the most recent action | `<latestAction><text>` |
| `policyArea` | Congress.gov policy area classification | `<policyArea><name>` |
| `historySize` | Byte/character size of the full BILLSTATUS XML file | File size |
| `summaryLength` | Character length of the CRS summary (HTML-formatted) | `len(<summaries><text>)` |
| `actionCount` | Number of recorded legislative actions | Count of `<actions><item>` |
| `versionCount` | Number of available bill text versions | Count of `<textVersions>` entries |
| `budgetEstimateCount` | Number of CBO cost estimates | Count of `<cboCostEstimates>` entries |
| `amendmentCount` | Number of amendments filed | Count of `<amendments>` entries |
| `relatedBillsCount` | Number of related bills identified by Congress.gov | Count of `<relatedBills><item>` |
| `committeeCount` | Number of committee referrals | Count of `<committees><item>` |
| `sponsorCount` | Number of co-sponsors (may exclude primary sponsor) | Count of `<cosponsors>` entries |
| `congress` | Congress number (e.g. `118`) | `<congress>` |
| `chamber` | Originating chamber (`House` or `Senate`) | `<originChamber>` |

**Appropriations filter:** `title.str.contains('Appropriation', case=False)` yields **953 bills**.


In [16]:
bbi_df.isnull().sum()

id                       0
title                    0
introducedDate           0
lastActionDate           5
daysActive               5
status                   5
policyArea             875
historySize              0
summaryLength            0
actionCount              0
versionCount             0
budgetEstimateCount      0
amendmentCount           0
relatedBillsCount        0
committeeCount           0
sponsorCount             0
dtype: int64

In [17]:
# Year coverage
print(bbi_df["introducedDate"].min(), bbi_df["introducedDate"].max())

# Extract congress + bill type
bbi_df[["congress", "bill_type"]] = bbi_df["id"].str.extract(r"^(\d+)-([a-z]+)-")
print(bbi_df["congress"].value_counts().sort_index())
print(bbi_df["bill_type"].value_counts())

2011-01-05 2026-05-22
congress
112    12299
113    10637
114    12063
115    13556
116    16601
117    17828
118    19315
119    16241
Name: count, dtype: int64
bill_type
hr         64879
s          34808
hres        9528
sres        6001
hjres       1142
hconres     1098
sjres        675
sconres      409
Name: count, dtype: int64


- hr — House bills
- s — Senate bills
- hres — House resolutions
- sres — Senate resolutions
- hjres — House joint resolutions
- sjres — Senate joint resolutions
- hconres, sconres — concurrent resolutions

In [18]:
approp_df = bbi_df[bbi_df["title"].str.contains("Appropriation", case=False)]
print(len(approp_df))

953


In [19]:
# Save the list appropriations bills to a CSV file
os.makedirs("data", exist_ok=True)
approp_df.to_csv("data/appropriations_bills.csv", index=False)

## Explore Single Bill / Node Structure (XML parsing)

In [20]:
xml_path = Path("../../../bills/118-hr-4366/1_reported-in-house.xml")
tree = normalize_bill(xml_path)
df_financial = build_financial_df(tree)
check_coverage(df_financial, tree)
df_financial.head()

✓  All 72 dollar-amount node occurrences represented (108 rows)


,node_idx,account,level,type,amount,needs_review,preview,body_text
0,4,TITLE I—DEPARTMENT OF DEFENSE > Military const...,primary,appropriation,1.517455e+09,False,"For acquisition, construction, installation, a...","For acquisition, construction, installation, a..."
1,4,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,cap,3.457750e+08,False,", of this amount, not to exceed $345,775,000 s...","For acquisition, construction, installation, a..."
2,4,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,earmark,1.629000e+08,False,of the amount made available under this headin...,"For acquisition, construction, installation, a..."
3,5,TITLE I—DEPARTMENT OF DEFENSE > Military const...,primary,appropriation,4.477961e+09,False,"For acquisition, construction, installation, a...","For acquisition, construction, installation, a..."
4,5,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,cap,6.026250e+08,False,", of this amount, not to exceed $602,625,000 s...","For acquisition, construction, installation, a..."


In [21]:
tree.nodes[0].__dir__()

['match_path',
 'display_path',
 'tag',
 'element_id',
 'header_text',
 'body_text',
 'section_number',
 'division_label',
 'division_key',
 'display_text',
 'body_index',
 '__module__',
 '__annotations__',
 '__doc__',
 '__dict__',
 '__weakref__',
 '__dataclass_params__',
 '__dataclass_fields__',
 '__init__',
 '__repr__',
 '__eq__',
 '__setattr__',
 '__delattr__',
 '__hash__',
 '__match_args__',
 '__new__',
 '__str__',
 '__getattribute__',
 '__lt__',
 '__le__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__getstate__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [30]:
import pandas as pd

# Full tree as a DataFrame — the fastest way to see all fields at once
df_tree = pd.DataFrame(
    [{**vars(n), "display_path": list(n.display_path), "match_path": list(n.match_path)} for n in tree.nodes]
)
df_tree.head(20)

,match_path,display_path,tag,element_id,header_text,body_text,section_number,division_label,division_key,display_text,body_index
0,"[front matter, masthead]",[],front-matter,front-matter-masthead,,118th CONGRESS\n1st Session\nH. R. 4366\nA BILL,,,,,0
1,"[front matter, official title]",[],front-matter,front-matter-official-title,,Making appropriations for military constructio...,,,,,0
2,"[front matter, enacting clause]",[],front-matter,front-matter-enacting-clause,,Be it enacted by the Senate and House of Repre...,,,,,0
3,[],[],section,ID79993566F894480FBAE9AE6C63CC75EA,,"That the following sums are appropriated, out ...",,,,"That the following sums are appropriated, out ...",0
4,"[department of defense, military construction,...","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",appropriations-intermediate,HF975A2F8EB9D45459DD00B72CA4AC07F,"Military construction, army","For acquisition, construction, installation, a...",,,,"For acquisition, construction, installation, a...",0
5,"[department of defense, military construction,...","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",appropriations-intermediate,H9446F39D3F8A49AAA485227C7ACBA3DF,"Military construction, navy and marine corps","For acquisition, construction, installation, a...",,,,"For acquisition, construction, installation, a...",0
6,"[department of defense, military construction,...","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",appropriations-intermediate,HE5E45708B3A34B3A8D1AF70E6BF0F87A,"Military construction, air force","For acquisition, construction, installation, a...",,,,"For acquisition, construction, installation, a...",0
7,"[department of defense, military construction,...","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",appropriations-small,HE03578B2A5FA40A0AF61E26D6F0DD6A5,(INCLUDING TRANSFER OF FUNDS),"For acquisition, construction, installation, a...",,,,"For acquisition, construction, installation, a...",0
8,"[department of defense, military construction,...","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",appropriations-intermediate,H1BDF3A160C924225B87BA920F6538797,"Military construction, army national guard","For construction, acquisition, expansion, reha...",,,,"For construction, acquisition, expansion, reha...",0
9,"[department of defense, military construction,...","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",appropriations-intermediate,H7E8B2BE53C8F43D9BD73B2D241397FC6,"Military construction, air national guard","For construction, acquisition, expansion, reha...",,,,"For construction, acquisition, expansion, reha...",0


In [28]:
# What tags exist, and how common are they?
df_tree["tag"].value_counts()

tag
section                        116
subsection                      37
appropriations-small            29
appropriations-intermediate     20
front-matter                     3
Name: count, dtype: int64

In [26]:
# For appropriations nodes only — compare header_text vs display_path side by side
appro = df_tree[df_tree["tag"].str.startswith("appro")]
appro[["tag", "header_text", "display_path", "division_label", "section_number"]].head(30)

,tag,header_text,display_path,division_label,section_number
4,appropriations-intermediate,"Military construction, army","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
5,appropriations-intermediate,"Military construction, navy and marine corps","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
6,appropriations-intermediate,"Military construction, air force","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
7,appropriations-small,(INCLUDING TRANSFER OF FUNDS),"[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
8,appropriations-intermediate,"Military construction, army national guard","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
9,appropriations-intermediate,"Military construction, air national guard","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
10,appropriations-intermediate,"Military construction, army reserve","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
11,appropriations-intermediate,"Military construction, navy reserve","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
12,appropriations-intermediate,"Military construction, air force reserve","[TITLE I—DEPARTMENT OF DEFENSE, Military const...",,
13,appropriations-intermediate,Security investment program,"[TITLE I—DEPARTMENT OF DEFENSE, Security inves...",,


In [33]:
# How deep is display_path for each tag type?
df_tree["path_depth"] = df_tree["display_path"].apply(len)
df_tree.groupby("tag")["path_depth"].value_counts()

tag                          path_depth
appropriations-intermediate  2             19
                             3              1
appropriations-small         4             27
                             2              2
front-matter                 0              3
section                      4             61
                             3             54
                             0              1
subsection                   5             23
                             4             14
Name: count, dtype: int64

In [34]:
df_tree[df_tree["path_depth"] > 2]

,match_path,display_path,tag,element_id,header_text,body_text,section_number,division_label,division_key,display_text,body_index,path_depth
24,"[department of defense, administrative provisi...","[TITLE I—DEPARTMENT OF DEFENSE, Administrative...",section,H017245F553FE47D88482683C51E5ED0D,,None of the funds made available in this title...,Sec. 101,,,None of the funds made available in this title...,0,3
25,"[department of defense, administrative provisi...","[TITLE I—DEPARTMENT OF DEFENSE, Administrative...",section,H72FBE46FB44D4EA68EF1FB1BA09C274D,,Funds made available in this title for constru...,Sec. 102,,,Funds made available in this title for constru...,0,3
26,"[department of defense, administrative provisi...","[TITLE I—DEPARTMENT OF DEFENSE, Administrative...",section,HBD31C73356934DB39299F034567F4C88,,Funds made available in this title for constru...,Sec. 103,,,Funds made available in this title for constru...,0,3
27,"[department of defense, administrative provisi...","[TITLE I—DEPARTMENT OF DEFENSE, Administrative...",section,H389F4E1429C24EA98EAA1E663CE8F247,,None of the funds made available in this title...,Sec. 104,,,None of the funds made available in this title...,0,3
28,"[department of defense, administrative provisi...","[TITLE I—DEPARTMENT OF DEFENSE, Administrative...",section,H106B440A17504091A2E2E939BC18883D,,None of the funds made available in this title...,Sec. 105,,,None of the funds made available in this title...,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...
199,"[general provisions, sec. 416, (a) in general]","[TITLE IV, GENERAL PROVISIONS, sec. 416, (a) I...",subsection,H7D5B3EB873EC47E5A1B6EAD02C50A4BB,In general,(a)In general Notwithstanding section 7 of tit...,Sec. 416,,,(a) In general Notwithstanding section 7 of ti...,0,4
200,"[general provisions, sec. 416, (b) discriminat...","[TITLE IV, GENERAL PROVISIONS, sec. 416, (b) D...",subsection,H62F491C37CF74D1A86E6AF33172FD429,Discriminatory action defined,(b)Discriminatory action defined.—As used in s...,Sec. 416,,,(b) Discriminatory action defined.—As used in ...,0,4
201,"[general provisions, sec. 416, (c) accreditati...","[TITLE IV, GENERAL PROVISIONS, sec. 416, (c) A...",subsection,HEC977D20B35041D7A9B3E5C366C1267A,Accreditation; Licensure; Certification,(c)Accreditation; Licensure; Certification.—Th...,Sec. 416,,,(c) Accreditation; Licensure; Certification.—T...,0,4
202,"[general provisions, sec. 417]","[TITLE IV, GENERAL PROVISIONS, sec. 417]",section,HC9F37089B18C4C41B3A38FB8A3E81A6B,,None of the funds made available by this Act m...,Sec. 417,,,None of the funds made available by this Act m...,0,3


In [40]:
df_tree["display_path"].apply(tuple).unique()

array([(),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, army'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, navy and marine corps'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, air force'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, defense-Wide'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, army national guard'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, air national guard'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, army reserve'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, navy reserve'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Military construction, air force reserve'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Security investment program'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Department of defense base closure account'),
       ('TITLE I—DEPARTMENT OF DEFENSE', 'Family housing construction, army'),
    

## PDF parsing

In [42]:
pdf_path = Path("../../../tests/corpus/118-hr-4366/1_reported-in-house.pdf")
pages = extract_clean_pages(pdf_path)

# Page-level overview
print(f"{len(pages)} pages")

# All lines as a flat DataFrame
df_lines = pd.DataFrame(
    [
        {
            "page": page.page_number,
            "line_number": ln.line_number,
            "glyph_size": ln.glyph_size,
            "content_left": ln.geom.content_left if ln.geom else None,
            "text": ln.text,
        }
        for page in pages
        for ln in page.lines
    ]
)
df_lines.head(40)

94 pages


,page,line_number,glyph_size,content_left,text
0,1,NaN,NaN,NaN,IB
1,1,NaN,NaN,NaN,Union Calendar No. 94
2,1,NaN,NaN,NaN,118TH CONGRESS
3,1,NaN,NaN,NaN,1ST SESSION H. R. 4366
4,1,NaN,NaN,NaN,[Report No. 118–122]
5,1,NaN,NaN,NaN,Making appropriations for military constructio...
6,1,NaN,NaN,NaN,"Affairs, and related agencies for the fiscal y..."
7,1,NaN,NaN,NaN,"2024, and for other purposes."
8,1,NaN,NaN,NaN,IN THE HOUSE OF REPRESENTATIVES
9,1,NaN,NaN,NaN,"JUNE 27, 2023"


In [43]:
anchors = extract_anchors(pages)

df_anchors = pd.DataFrame(
    [
        {
            "kind": a.kind,  # title | section | account | major | agency | grouping | subsection | preamble
            "page": a.page_number,
            "line": a.line_number,
            "division": a.division,
            "text": a.text,
            "breadcrumb": breadcrumb_for(a, anchors),  # equivalent of display_path
        }
        for a in anchors
    ]
)

# What kinds exist and how many?
df_anchors["kind"].value_counts()

kind
section       115
account        48
agency         18
title           4
major           4
grouping        4
subsection      4
Name: count, dtype: int64

In [44]:
from deltatrack.parsers.pdf_text import extract_clean_pages  # noqa: F401

v1_pages = extract_clean_pages(Path("../tests/corpus/118-hr-4366/1_reported-in-house.pdf"))
v2_pages = extract_clean_pages(Path("../tests/corpus/118-hr-4366/2_engrossed-in-house.pdf"))
pdf_diff = diff_pdfs(v1_pages, v2_pages)

df_hunks = pd.DataFrame(
    [
        {
            "change_type": h.change_type,
            "v1_anchor": h.v1_anchor.text if h.v1_anchor else None,
            "v2_anchor": h.v2_anchor.text if h.v2_anchor else None,
            "v1_breadcrumb": breadcrumb_for(h.v1_anchor, pdf_diff.v1_anchors) if h.v1_anchor else None,
            "v2_breadcrumb": breadcrumb_for(h.v2_anchor, pdf_diff.v2_anchors) if h.v2_anchor else None,
        }
        for h in pdf_diff.hunks
    ]
)
df_hunks

FileNotFoundError: D:\ruggbk\GitHub\AgoraDMV\DeltaTrack\docs\research\tests\corpus\118-hr-4366\1_reported-in-house.pdf

In [ ]:
pdf_diff.hunks[0].__dir__()

In [ ]:
pdf_diff.hunks[0].v1_text

### Compare the text of the first five hunks (pdf) vs first five nodes (xml)

In [ ]:
for h in pdf_diff.hunks[0:5]:
    print(h.v1_text)

In [ ]:
for n in tree.nodes[0:5]:
    print(n.body_text)